# Video Category Pipeline

- Reference data
- One-time extraction
- Raw JSON → Volume
- Volume → Bronze Delta

In [0]:
# %pip install google-api-python-client python-dotenv

# %restart_python

## 1. Create Volume and incoming folder

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS youtube_content_intelligence.bronze.vol_video_category;

In [0]:
dbutils.fs.mkdirs("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/")

## 2. Extract video category data

In [0]:
from src.extraction.video_category import extract_video_categories

categories = extract_video_categories()

## 3. Write raw JSON to Volume

In [0]:
import json

volume_path = "/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/video_category.json"

with open(volume_path, "w") as file:
    json.dump(categories, file, indent=2)

## 4. Verify raw JSON

In [0]:
display(dbutils.fs.ls("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/"))

In [0]:
display(dbutils.fs.head("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/video_category.json", 1000))

## 5. Load raw JSON

In [0]:
df = (
    spark.read
    .option("multiLine", "true")
    .json("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/video_category.json")
)

In [0]:
df.printSchema()

Alternative explicit schema definition:

In [0]:
# from pyspark.sql.types import StructType, StructField, StringType, BooleanType, ArrayType

# schema = StructType([
#     StructField("kind", StringType(), True),
#     StructField("etag", StringType(), True),
#     StructField("items", ArrayType(
#         StructType([
#             StructField("kind", StringType(), True),
#             StructField("etag", StringType(), True),
#             StructField("id", StringType(), True),
#             StructField("snippet", StructType([
#                 StructField("title", StringType(), True),
#                 StructField("assignable", BooleanType(), True),
#                 StructField("channelId", StringType(), True)
#             ]), True)
#         ])
#     ), True)
# ])

# df = (
#     spark.read
#     .option("multiLine", "true")
#     .schema(schema)
#     .json("/Volumes/youtube_content_intelligence/bronze/vol_video_category/incoming/video_category.json")
# )

## 6. Write to Bronze Delta

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("youtube_content_intelligence.bronze.brz_video_category")

## 7. Validate Bronze table

In [0]:
%sql
SELECT *
FROM youtube_content_intelligence.bronze.brz_video_category;

In [0]:
%sql
DESCRIBE TABLE youtube_content_intelligence.bronze.brz_video_category;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM youtube_content_intelligence.bronze.brz_video_category;